
# Sales Forecasting Using Machine Learning

**Project:** CognoRise Infotech — Machine Learning Internship  
**Goal:** Explore monthly sales, compare simple forecasting baselines with a Random Forest model, evaluate them on a chronological holdout, and forecast the next 12 months.

## Important notes
- This notebook uses the **same public monthly sales dataset URL as the original notebook** (`a10.csv`). It is a public example dataset, not private company sales data.
- The final 12 months are held out for evaluation. Test features use historical observed values, representing a rolling one-month-ahead forecast evaluation.
- The next-12-month forecast uses the method with the lowest MAE on that holdout. A flat forecast can be valid for the last-value baseline, so interpret it alongside the comparison table and plot rather than assuming every forecast must rise or fall.
- Run all cells from top to bottom in a fresh Colab session before submitting.


## 1. Import libraries

In [ ]:

import os
import zipfile
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
OUTPUT_DIR = "sales_forecasting_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries imported successfully.")


## 2. Load and validate the dataset

In [ ]:

DATA_URL = "https://raw.githubusercontent.com/selva86/datasets/master/a10.csv"
raw_df = pd.read_csv(DATA_URL)

print("Original columns:", raw_df.columns.tolist())
# This project expects the original dataset's date and value columns.
if not {"date", "value"}.issubset(raw_df.columns):
    raise ValueError(
        "Expected columns 'date' and 'value'. Please check the dataset URL and column names."
    )

data = raw_df[["date", "value"]].copy()
data["date"] = pd.to_datetime(data["date"], errors="coerce")
data["value"] = pd.to_numeric(data["value"], errors="coerce")
data = data.dropna(subset=["date", "value"])
data = data.sort_values("date")

# If the source has more than one record for a date, aggregate the sales values.
data = data.groupby("date", as_index=False)["value"].sum()
data = data.sort_values("date").reset_index(drop=True)

if len(data) <= 24:
    raise ValueError("Not enough observations for 12 lag features and a 12-month test period.")

print(f"Rows: {len(data)}")
print(f"Date range: {data['date'].min().date()} to {data['date'].max().date()}")
print("Missing values:", data.isna().sum())
display(data.head())
display(data.tail())


## 3. Explore the sales time series

In [ ]:

plt.figure(figsize=(12, 5))
plt.plot(data["date"], data["value"], marker="o", linewidth=1.5)
plt.title("Monthly Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Sales value")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sales_historical_trend.png"), dpi=150)
plt.show()

print("Descriptive statistics:")
display(data["value"].describe().to_frame("Sales value"))


## 4. Create lag features and a chronological train/test split

In [ ]:

# lag_1 = previous observed month; lag_12 = observed value from 12 months earlier.
model_data = data.copy()
FEATURES = [f"lag_{i}" for i in range(1, 13)]
for i in range(1, 13):
    model_data[f"lag_{i}"] = model_data["value"].shift(i)

model_data = model_data.dropna().reset_index(drop=True)

# Keep the latest 12 rows as a holdout; never shuffle a time series.
HOLDOUT_MONTHS = 12
split_index = len(model_data) - HOLDOUT_MONTHS
train_df = model_data.iloc[:split_index].copy()
test_df = model_data.iloc[split_index:].copy()

X_train, y_train = train_df[FEATURES], train_df["value"]
X_test, y_test = test_df[FEATURES], test_df["value"]

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Test period:", test_df["date"].min().date(), "to", test_df["date"].max().date())
display(model_data.head())


## 5. Train models and compare results

In [ ]:

# Model 1: Random Forest regressor
rf_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

# Baseline 1: next month equals the most recently observed month.
last_value_predictions = test_df["lag_1"].to_numpy()

# Baseline 2: next month equals the value observed 12 months earlier.
seasonal_naive_predictions = test_df["lag_12"].to_numpy()

predictions_by_model = {
    "Random Forest": np.asarray(rf_predictions),
    "Last-value baseline": np.asarray(last_value_predictions),
    "Seasonal-naive baseline": np.asarray(seasonal_naive_predictions),
}

def calculate_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": r2_score(actual, predicted),
    }

comparison_rows = []
for model_name, model_predictions in predictions_by_model.items():
    row = {"Model": model_name}
    row.update(calculate_metrics(y_test, model_predictions))
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).sort_values("MAE").reset_index(drop=True)
best_model_name = comparison_df.loc[0, "Model"]
print("Model comparison on the latest 12 months (rolling one-month-ahead evaluation):")
display(comparison_df.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}"}))
print("Selected method based on lowest holdout MAE:", best_model_name)
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "sales_model_comparison.csv"), index=False)


## 6. Plot actual vs predicted sales on the holdout

In [ ]:

plt.figure(figsize=(12, 5))
plt.plot(test_df["date"], y_test.to_numpy(), marker="o", label="Actual sales", linewidth=2)
for name, values in predictions_by_model.items():
    plt.plot(test_df["date"], values, marker="o", linestyle="--", label=name)
plt.title("Actual vs Predicted Sales — 12-Month Holdout")
plt.xlabel("Date")
plt.ylabel("Sales value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sales_forecast_comparison.png"), dpi=150)
plt.show()


## 7. Forecast the next 12 months

In [ ]:

# Forecast with the method selected from holdout MAE. Recursive forecasting is used
# when generating a multi-month future horizon, because future actual values are unknown.
history = data["value"].astype(float).tolist()
last_observed_date = data["date"].max()
future_dates = pd.date_range(
    start=last_observed_date + pd.offsets.MonthBegin(1),
    periods=12,
    freq="MS",
)
future_values = []

for future_date in future_dates:
    if best_model_name == "Random Forest":
        row = {f"lag_{i}": history[-i] for i in range(1, 13)}
        X_future = pd.DataFrame([row], columns=FEATURES)
        prediction = float(rf_model.predict(X_future)[0])
    elif best_model_name == "Seasonal-naive baseline":
        prediction = float(history[-12])
    else:  # Last-value baseline
        prediction = float(history[-1])

    future_values.append(prediction)
    history.append(prediction)

forecast_df = pd.DataFrame({"date": future_dates, "Forecast_Sales": future_values})
forecast_path = os.path.join(OUTPUT_DIR, "sales_forecast_next_12_months.csv")
forecast_df.to_csv(forecast_path, index=False)

print("Selected forecast method:", best_model_name)
print("Future 12-month forecast:")
display(forecast_df)

plt.figure(figsize=(12, 5))
recent_history = data.tail(min(36, len(data)))
plt.plot(recent_history["date"], recent_history["value"], marker="o", label="Historical sales")
plt.plot(forecast_df["date"], forecast_df["Forecast_Sales"], marker="o", linestyle="--", label="Forecast")
plt.axvline(last_observed_date, color="gray", linestyle=":", label="Forecast starts")
plt.title("Next 12 Months Sales Forecast")
plt.xlabel("Date")
plt.ylabel("Sales value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sales_forecast_future.png"), dpi=150)
plt.show()

if np.allclose(forecast_df["Forecast_Sales"].to_numpy(), forecast_df["Forecast_Sales"].iloc[0]):
    print("Note: The selected forecasting method produced a flat forecast. This can happen with a last-value baseline or a Random Forest recursive forecast; assess it using the holdout metrics and historical context.")


## 8. Save the model bundle and cleaned data

In [ ]:

data.to_csv(os.path.join(OUTPUT_DIR, "sales_cleaned.csv"), index=False)

model_bundle = {
    "selected_method": best_model_name,
    "random_forest_model": rf_model,
    "feature_columns": FEATURES,
    "holdout_metrics": comparison_df.to_dict(orient="records"),
    "last_12_observed_values": data["value"].tail(12).tolist(),
    "last_observed_date": str(last_observed_date.date()),
    "forecast_horizon_months": 12,
    "dataset_url": DATA_URL,
}
joblib.dump(model_bundle, os.path.join(OUTPUT_DIR, "sales_forecasting_model.pkl"))

print("Saved files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)


## 9. Export project outputs as a ZIP

In [ ]:

zip_path = "Sales_Forecasting_Project_Files.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    for filename in os.listdir(OUTPUT_DIR):
        file_path = os.path.join(OUTPUT_DIR, filename)
        if os.path.isfile(file_path):
            zip_file.write(file_path, arcname=filename)

print(f"Created {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Google Colab. Download the ZIP from the file browser if available.")



## 10. Conclusion

This project compares a Random Forest model with last-value and seasonal-naive baselines using a chronological 12-month holdout. The method with the lowest holdout MAE is used for the 12-month recursive forecast. MAE and RMSE measure prediction error (lower is better); R² measures explained variation and can be negative on a difficult holdout. A forecast may become flat, especially for the last-value baseline or recursive tree models, so the forecast should be interpreted together with the comparison table and historical graph.

**Before submission:** run all cells from top to bottom, inspect the forecast table and plots, and use the actual printed metric values in the repository README. Do not enter metric values manually.
